# 00 - Configuración y catálogo de datos

Este notebook comprueba qué ficheros hay en `data/processed` y detecta si cada JSON pertenece a Radon o a PyCEFR.

La idea es empezar sin asumir demasiado sobre los nombres de los archivos.

In [22]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVEL_ORDER = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
LEVEL_ORDER_INV = {v: k for k, v in LEVEL_ORDER.items()}
RADON_RANK_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}
RADON_RANK_ORDER_INV = {v: k for k, v in RADON_RANK_ORDER.items()}

def clean_file_name(path):
    """Devuelve un nombre de fichero comparable entre herramientas."""
    return Path(str(path)).name

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def detect_csv_kind(path):
    """
    Intenta detectar si un CSV corresponde a PyCEFR o a Radon
    mirando el nombre del archivo y sus columnas.
    """
    name = path.name.lower()

    # Primero usamos el nombre como pista rápida
    if "radon" in name:
        return "radon_csv"

    if "pycefr" in name or name == "data.csv" or name == "data_pycefr.csv":
        possible_by_name = "pycefr_csv"
    else:
        possible_by_name = "csv_unknown"

    # Después intentamos leer columnas
    try:
        df_sample = pd.read_csv(path, nrows=5)
        columns = set(df_sample.columns)
    except Exception:
        return possible_by_name

    # Columnas típicas de PyCEFR
    pycefr_columns = {"Class", "Start Line", "End Line", "Displacement", "Level"}
    if pycefr_columns.intersection(columns):
        return "pycefr_csv"

    # Columnas típicas de Radon
    radon_columns = {"Repository", "File Name", "Type", "Name", "Line", "Complexity", "Rank"}
    if radon_columns.intersection(columns):
        return "radon_csv"

    return possible_by_name


def detect_file_kind(path):
    """
    Clasifica cualquier archivo de resultados.
    """
    suffix = path.suffix.lower()

    if suffix == ".json":
        json_kind = detect_json_kind(path)
        if json_kind == "pycefr":
            return "pycefr_json"
        if json_kind == "radon":
            return "radon_json"
        return "json_unknown"

    if suffix == ".csv":
        return detect_csv_kind(path)

    return "unknown"

## Carpetas esperadas

La estructura recomendada es:

```text
data/
  processed/
    caso_1/
    caso_2/
    caso_3/
```

Cada carpeta de caso puede contener los JSON o CSV generados por las herramientas.

In [23]:
print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR existe:", PROCESSED_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: /home/juan/Documents/Analisis-CC-PyCEFR
PROCESSED_DIR existe: True
OUTPUT_DIR: /home/juan/Documents/Analisis-CC-PyCEFR/outputs


In [24]:
def find_case_folders():
    """
    Devuelve las carpetas de casos dentro de data/processed.
    Cada subcarpeta se considera un caso.
    """
    if not PROCESSED_DIR.exists():
        return []

    return sorted([p for p in PROCESSED_DIR.iterdir() if p.is_dir()])


In [26]:
case_folders = find_case_folders()
print("Número de proyectos a analizar:", len(case_folders))
for p in case_folders:
    print("-", p.name)

Número de proyectos a analizar: 1
- python-beginner-programming-exercises


In [27]:
rows = []

for case_dir in case_folders:
    for path in sorted(case_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in [".json", ".csv"]:
            kind = detect_file_kind(path)

            rows.append({
                "case": case_dir.name,
                "file": path.name,
                "path": str(path),
                "extension": path.suffix.lower(),
                "detected_kind": kind,
                "size_kb": round(path.stat().st_size / 1024, 2),
            })

catalog = pd.DataFrame(rows)
catalog

,case,file,path,extension,detected_kind,size_kb
0,python-beginner-programming-exercises,data_pycefr.csv,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...,.csv,pycefr_csv,67.03
1,python-beginner-programming-exercises,data_pycefr.json,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...,.json,pycefr_json,242.52
2,python-beginner-programming-exercises,radon_output_python-beginner-programming-exerc...,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...,.csv,radon_csv,10.03
3,python-beginner-programming-exercises,radon_output_python-beginner-programming-exerc...,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...,.json,radon_json,38.13


In [28]:
catalog.to_csv(OUTPUT_DIR / "00_catalogo_ficheros.csv", index=False)

summary_catalog = (
    catalog
    .groupby(["case", "detected_kind"])
    .size()
    .reset_index(name="n_files")
)

summary_catalog

,case,detected_kind,n_files
0,python-beginner-programming-exercises,pycefr_csv,1
1,python-beginner-programming-exercises,pycefr_json,1
2,python-beginner-programming-exercises,radon_csv,1
3,python-beginner-programming-exercises,radon_json,1


In [29]:
catalog.to_csv(OUTPUT_DIR / "00_catalogo_ficheros.csv", index=False)

summary_catalog = (
    catalog
    .groupby(["case", "detected_kind"])
    .size()
    .reset_index(name="n_files")
)

summary_catalog

,case,detected_kind,n_files
0,python-beginner-programming-exercises,pycefr_csv,1
1,python-beginner-programming-exercises,pycefr_json,1
2,python-beginner-programming-exercises,radon_csv,1
3,python-beginner-programming-exercises,radon_json,1


In [31]:
#Comprobando casos para vetr qure funciona bien el notebook 
case_status = (
    catalog
    .groupby("case")
    .agg(
        n_files=("file", "count"),
        has_pycefr=("detected_kind", lambda x: any(k.startswith("pycefr") for k in x)),
        has_radon=("detected_kind", lambda x: any(k.startswith("radon") for k in x)),
        has_pycefr_json=("detected_kind", lambda x: "pycefr_json" in set(x)),
        has_radon_json=("detected_kind", lambda x: "radon_json" in set(x)),
        has_pycefr_csv=("detected_kind", lambda x: "pycefr_csv" in set(x)),
        has_radon_csv=("detected_kind", lambda x: "radon_csv" in set(x)),
    )
    .reset_index()
)

case_status["is_comparable"] = case_status["has_pycefr"] & case_status["has_radon"]

case_status

,case,n_files,has_pycefr,has_radon,has_pycefr_json,has_radon_json,has_pycefr_csv,has_radon_csv,is_comparable
0,python-beginner-programming-exercises,4,True,True,True,True,True,True,True


## Comentario para la memoria

En esta fase se realiza un catálogo inicial de los resultados generados por las herramientas. Esto permite comprobar qué casos del corpus tienen salida de Radon, salida de PyCEFR o ficheros adicionales. Esta comprobación es importante porque evita comparar resultados incompletos o mal ubicados.